# Deepfake Detection Notebook (Easy Run)
Is notebook me aap step-by-step ya one-click mode me pura pipeline chala sakte ho.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.environ['PYTHONPATH'] = str(SRC)
from deepfake_detector.utils.notebook_helpers import run_cmd, show_manifest_counts, show_sample_counts, ensure_paths
print('Project root:', ROOT)

## Parameters (Yahin change karo)

In [ ]:
RAW_ROOT = 'data/raw'
PROCESSED_ROOT = 'data/processed'
MANIFEST_PATH = f'{PROCESSED_ROOT}/manifest.json'
IMGVID_SAMPLES = f'{PROCESSED_ROOT}/image_video_samples.json'
AUDIO_SAMPLES = f'{PROCESSED_ROOT}/audio_samples.json'
IMAGE_MODEL = 'models/exports/image_tf_model.keras'
VIDEO_MODEL = 'models/checkpoints/video_gru.pt'
AUDIO_MODEL = 'models/exports/audio_rf.joblib'
EPOCHS = 5
RUN_IMAGE = True
RUN_VIDEO = True
RUN_AUDIO = True

## 1) Build Dataset Manifest

In [ ]:
run_cmd(f'python -m deepfake_detector.data.dataset_manifest --raw-root {RAW_ROOT} --out {MANIFEST_PATH}')
show_manifest_counts(MANIFEST_PATH)

## 2) Preprocess Image/Video + Audio

In [ ]:
run_cmd(f'python -m deepfake_detector.data.preprocess_image_video --manifest {MANIFEST_PATH} --out-root {PROCESSED_ROOT}')
run_cmd(f'python -m deepfake_detector.data.preprocess_audio --manifest {MANIFEST_PATH} --out-root {PROCESSED_ROOT}')
show_sample_counts(IMGVID_SAMPLES)
show_sample_counts(AUDIO_SAMPLES)

## 3) Train Models

In [ ]:
if RUN_IMAGE:
    run_cmd(f'python -m deepfake_detector.train --modality image --samples-json {IMGVID_SAMPLES} --out {IMAGE_MODEL} --epochs {EPOCHS}')
if RUN_VIDEO:
    run_cmd(f'python -m deepfake_detector.train --modality video --samples-json {IMGVID_SAMPLES} --out {VIDEO_MODEL} --epochs {EPOCHS}')
if RUN_AUDIO:
    run_cmd(f'python -m deepfake_detector.train --modality audio --samples-json {AUDIO_SAMPLES} --out {AUDIO_MODEL}')

## 4) Evaluate

In [ ]:
if RUN_IMAGE:
    run_cmd(f'python -m deepfake_detector.evaluate --modality image --samples-json {IMGVID_SAMPLES} --model {IMAGE_MODEL}')
if RUN_VIDEO:
    run_cmd(f'python -m deepfake_detector.evaluate --modality video --samples-json {IMGVID_SAMPLES} --model {VIDEO_MODEL}')
if RUN_AUDIO:
    run_cmd(f'python -m deepfake_detector.evaluate --modality audio --samples-json {AUDIO_SAMPLES} --model {AUDIO_MODEL}')

## 5) Inference (Single Sample)
Niche `TEST_*` paths update karke run karo.

In [ ]:
TEST_IMAGE = 'path/to/sample.jpg'
TEST_VIDEO = 'path/to/sample.mp4'
TEST_AUDIO = 'path/to/sample.wav'
# ensure_paths([TEST_IMAGE, TEST_AUDIO])  # optional
cmd = f'python -m deepfake_detector.infer --image {TEST_IMAGE} --image-model {IMAGE_MODEL} --video {TEST_VIDEO} --video-model {VIDEO_MODEL} --audio {TEST_AUDIO} --audio-model {AUDIO_MODEL}'
print(cmd)
# run_cmd(cmd)

## One-Click Run (Optional)
Agar aap terminal se karna chahte ho to `scripts/run_pipeline.ps1` ya `scripts/start_notebook.ps1` use karo.